
# 02 GeoPandas: Tables That Know Where They Are

GeoPandas' single core idea: **it's a pandas DataFrame with one extra superpower** one column (called
the "geometry column") holds Shapely objects instead of numbers or text, and GeoPandas knows how to
filter, join, and analyze based on that column spatially, while every other column behaves exactly like
normal pandas.

If you already know pandas, you already know 80% of GeoPandas. If you don't know pandas at all, that's
worth a short detour first this notebook assumes basic familiarity (`.head()`, `.loc[]`, boolean
filtering).


In [ ]:

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon

# Building a GeoDataFrame directly, the way you'd build it from real parcel records
data = {
    "parcel_id": ["PARC-001", "PARC-002", "PARC-003"],
    "owner_name": ["Ekotto Land Holdings", "Mballa Family Trust", "Ekotto Land Holdings"],
    "price_xaf": [4_500_000, 6_200_000, 3_800_000],
    "status": ["sold", "available", "sold"],
    "geometry": [
        Polygon([(9.240, 4.150), (9.241, 4.150), (9.241, 4.151), (9.240, 4.151)]),
        Polygon([(9.250, 4.160), (9.251, 4.160), (9.251, 4.161), (9.250, 4.161)]),
        Polygon([(9.2405, 4.1505), (9.2415, 4.1505), (9.2415, 4.1515), (9.2405, 4.1515)]),
    ],
}

parcels = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:4326")
parcels



Notice three things that make this a *Geo*DataFrame rather than a plain DataFrame:

1. The `geometry=` argument tells GeoPandas which column holds the Shapely objects.
2. The `crs=` argument sets the **coordinate reference system** `EPSG:4326` is standard GPS lat/lon.
   This matters more than it looks like it does; the whole of notebook 03 is about why.
3. Every normal pandas operation still works filtering, grouping, `.head()` GeoPandas doesn't take
   any of that away, it adds to it.


In [ ]:

# Ordinary pandas filtering works exactly as expected
sold = parcels[parcels["status"] == "sold"]
sold


In [ ]:

# Plotting this is what "descartes" used to be needed for. It's built in now.
parcels.plot(column="status", legend=True, figsize=(5, 4))



## Reading and writing real files

GeoPandas reads/writes most geographic file formats (Shapefile, GeoJSON, GeoPackage) through one
function each under the hood this calls **Fiona** (or, in newer GeoPandas versions, the faster
**Pyogrio** engine) to do the actual file parsing. You almost never need to think about Fiona directly;
it's the plumbing, not something you call yourself.


In [ ]:

import tempfile
import os

# Writing to GeoJSON (a human-readable, widely-compatible geographic format)
parcels.to_file(os.path.join(tempfile.gettempdir(), "parcels.geojson"), driver="GeoJSON")

# Reading it back
reloaded = gpd.read_file(os.path.join(tempfile.gettempdir(), "parcels.geojson"))
reloaded



## Sample datasets with `geodatasets`

For practice (not for your real product data), the `geodatasets` package fetches well-known public
sample datasets so you're not always building tiny toy examples by hand. **This needs an internet
connection to download the data the first time** it will work on your own machine, but may not run in
every sandboxed environment.


In [ ]:

try:
    import geodatasets
    path = geodatasets.get_path("nybb")  # New York City boroughs a classic GeoPandas example dataset
    nyc = gpd.read_file(path)
    print(nyc.head())
    nyc.plot(figsize=(5, 5))
except Exception as e:
    print("Skipped this needs internet access to fetch the sample dataset.")
    print("On your own machine, this will work as-is. Error was:", e)



## Spatial joins GeoPandas' most powerful single feature

A **spatial join** merges two GeoDataFrames based on their geometric relationship instead of a shared ID
column this is the GeoPandas equivalent of `.intersects()`/`.contains()` from Shapely, but applied
across two entire tables at once instead of one pair of shapes at a time.


In [ ]:

# A "zones" GeoDataFrame representing administrative areas
zones = gpd.GeoDataFrame({
    "zone_name": ["North Buea", "South Buea"],
    "geometry": [
        Polygon([(9.230, 4.145), (9.245, 4.145), (9.245, 4.155), (9.230, 4.155)]),
        Polygon([(9.245, 4.155), (9.260, 4.155), (9.260, 4.165), (9.245, 4.165)]),
    ],
}, crs="EPSG:4326")

# sjoin finds, for every parcel, which zone (if any) it falls within
parcels_with_zone = gpd.sjoin(parcels, zones, how="left", predicate="within")
parcels_with_zone[["parcel_id", "owner_name", "zone_name"]]



This single `sjoin` call replaced what would otherwise be a manual loop checking every parcel against
every zone with `.within()` and GeoPandas does this efficiently using a spatial index under the hood
(built on Rtree see notebook 05) rather than a slow brute-force comparison.

## A quick early look at the spatial index

You don't need to build this yourself GeoPandas builds it automatically but it's worth knowing it
exists now, since notebook 05 explains why it matters for performance at real scale.


In [ ]:

# Every GeoDataFrame has a spatial index available on demand
idx = parcels.sindex
print(type(idx))
print("This index is what makes .sjoin() and large-scale queries fast more in notebook 05.")



## Exercises

### Exercise 1
Add a new parcel to the `parcels` GeoDataFrame any shape, any attributes using
`pd.concat()` (the standard pandas way to append rows; GeoPandas objects work the same way).


#### Solution

In [ ]:

new_parcel = gpd.GeoDataFrame({
    "parcel_id": ["PARC-004"],
    "owner_name": ["Nkemayang Estates"],
    "price_xaf": [5_100_000],
    "status": ["available"],
    "geometry": [Polygon([(9.255, 4.158), (9.256, 4.158), (9.256, 4.159), (9.255, 4.159)])],
}, crs="EPSG:4326")

parcels_updated = pd.concat([parcels, new_parcel], ignore_index=True)
parcels_updated


In [ ]:
# Your code here
another_parcel = gpd.GeoDataFrame({
    "parcel_id": ["PARC-005"],
    "owner_name": ["Tabot's Estate"],
    "price_xaf": [5_000_000],
    "status": ["available"],
    "geometry": [Polygon([(9.2405, 4.1505), (9.2415, 4.1505), (9.2415, 4.1515), (9.2405, 4.1515)])],
}, crs="EPSG:4326")

added_parcel = pd.concat([parcels, another_parcel], ignore_index=True)
added_parcel



### Exercise 2
Using ordinary pandas grouping (`.groupby()`), find the total `price_xaf` of parcels owned by
`"Ekotto Land Holdings"` in the original `parcels` GeoDataFrame. This is a reminder that GeoPandas
doesn't replace anything you already know from pandas it's additive.


In [ ]:
# Your code here


#### Solution

In [ ]:

totals = parcels.groupby("owner_name")["price_xaf"].sum()
print(totals["Ekotto Land Holdings"])



### Exercise 3
Write a function `parcels_in_zone(parcels_gdf, zones_gdf, zone_name)` that returns only the parcels
that fall within a named zone, using `gpd.sjoin()`.


In [ ]:
# Your code here


#### Solution

In [ ]:

def parcels_in_zone(parcels_gdf, zones_gdf, zone_name):
    joined = gpd.sjoin(parcels_gdf, zones_gdf, how="inner", predicate="within")
    return joined[joined["zone_name"] == zone_name]

parcels_in_zone(parcels, zones, "North Buea")



## What's next

You may have noticed the `.area` values on these geometries still don't mean anything in real-world
units (square meters, hectares) we're still working in raw lat/lon degrees. **`03_crs_and_pyproj.ipynb`**
fixes that: it's the notebook that makes your area and distance calculations actually correct, which
matters enormously for a product where "how big is this parcel" is a real financial and legal question.
